This notebook trains and applies the classifiers. For each feature set (AEF, OlmoEarth, both), it fits a linear probe (multinomial logistic regression) on training fields from every country, records the training and validation loss as it fits, and predicts the crop of every test field. Notebook 04 plots the results.

In [ ]:
import os
import time
import copy
import warnings
import numpy as np
import pandas as pd
import pyarrow as pa
from pathlib import Path
from sklearn.metrics import log_loss
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
warnings.filterwarnings('ignore',category=ConvergenceWarning)

In [ ]:
INTERIMDIR  = Path('../data/interim')
PROCDIR     = Path('../data/processed')
COLLECTIONS = pd.read_csv(INTERIMDIR/'collections.csv').collection.tolist()
FEATURESETS = {'aef':[f'aef{i:03d}' for i in range(128)],
               'olmoearth':[f'olmoearth{i:03d}' for i in range(128)],
               'both':[f'aef{i:03d}' for i in range(128)]+[f'olmoearth{i:03d}' for i in range(128)]}
TRAINFIELDS = 20000
VALFIELDS   = 5000
STEPITERS   = 10
MAXSTEPS    = 50
PATIENCE    = 5
PENALTY     = 1.0
SEED        = 42
NCORES      = len(os.sched_getaffinity(0))

pa.set_cpu_count(NCORES)
pa.set_io_thread_count(NCORES)

`TRAINFIELDS` and `VALFIELDS` are the fields sampled from each country's training and validation blocks. The probe is fitted in steps of `STEPITERS` optimiser iterations, up to `MAXSTEPS` steps. After each step, the training and validation loss are recorded, and fitting stops once the validation loss has not improved for `PATIENCE` steps. The weights from the best step are kept. `PENALTY` is the strength of the L2 penalty on the weights (scikit-learn's `C` is its inverse, so a larger `PENALTY` means smaller weights).

In [ ]:
def load_collection(collection):
    table  = pd.read_parquet(PROCDIR/f"fields_{collection.split('/')[1]}.parquet")
    blocks = table.chip.str[4:-4]+'_'+(table.chip.str[-4:-2].astype(int)//6).astype(str)+'_'+(table.chip.str[-2:].astype(int)//6).astype(str)
    mixed  = int((table.groupby(blocks).split.nunique()>1).sum())
    sample = pd.concat([table[table.split=='train'].sample(min(TRAINFIELDS,(table.split=='train').sum()),random_state=SEED),
                        table[table.split=='val'].sample(min(VALFIELDS,(table.split=='val').sum()),random_state=SEED),
                        table[table.split=='test']])
    return sample.assign(country=collection.split('/')[1]),{'blocks':blocks.nunique(),'mixedblocks':mixed}

def fit_probe(featureset,cols,train,val):
    trainweights,valweights = compute_sample_weight('balanced',train.crop),compute_sample_weight('balanced',val.crop)
    scaler  = StandardScaler().fit(train[cols])
    xtrain,xval = scaler.transform(train[cols]),scaler.transform(val[cols])
    model   = LogisticRegression(C=1/PENALTY,class_weight='balanced',max_iter=STEPITERS,warm_start=True)
    history,best,beststep = [],None,0
    for step in range(1,MAXSTEPS+1):
        model.fit(xtrain,train.crop)
        trainloss = log_loss(train.crop,model.predict_proba(xtrain),labels=model.classes_,sample_weight=trainweights)
        valloss   = log_loss(val.crop,model.predict_proba(xval),labels=model.classes_,sample_weight=valweights)
        history.append({'featureset':featureset,'step':step,'iterations':step*STEPITERS,'trainloss':trainloss,'valloss':valloss})
        if valloss<=min(row['valloss'] for row in history):
            best,beststep = copy.deepcopy(model),step
        if step-beststep>=PATIENCE:
            break
    return scaler,best,beststep,history

In [ ]:
loaded = [load_collection(collection) for collection in COLLECTIONS]
fields = pd.concat([sample for sample,_ in loaded],ignore_index=True)
blocks = pd.DataFrame([check for _,check in loaded])

print(f'Loaded {len(fields):,} fields from {fields.country.nunique()} countries...')
print(f'- {blocks.mixedblocks.sum()} of {blocks.blocks.sum():,} 6 km blocks have fields in more than one split (should be 0)')

In [ ]:
train     = fields[fields.split=='train']
val       = fields[(fields.split=='val')&fields.crop.isin(train.crop.unique())]
test      = fields.loc[fields.split=='test',['id','country','crop']].copy()
histories = []
print(f'Fitting one linear probe per feature set on {len(train):,} fields ({train.crop.nunique()} crops)...')

for featureset,cols in FEATURESETS.items():
    start = time.perf_counter()
    scaler,model,beststep,history = fit_probe(featureset,cols,train,val)
    test[featureset] = model.predict(scaler.transform(fields.loc[fields.split=='test',cols]))
    histories       += history
    minutes,seconds  = divmod(round(time.perf_counter()-start),60)
    print(f'- {featureset}: best validation loss at step {beststep} of {len(history)}, test accuracy {(test.crop==test[featureset]).mean():.3f} ({minutes}m {seconds}s)')

test.to_parquet(PROCDIR/'results_predictions.parquet',index=False)
pd.DataFrame(histories).to_csv(PROCDIR/'results_losses.csv',index=False)
print(f"Saved predictions to {PROCDIR/'results_predictions.parquet'} and losses to {PROCDIR/'results_losses.csv'}")